# Feature Engineering — Création de la Base Membres

Ce notebook consolide et enrichit les données individuelles des contacts CRM (`System ID` / `Relation ID`) pour préparer la modélisation de la rétention de l'Union des Marques.


### 1. Périmètre & Sources de données

| Source | Clé de jointure | Granularité source | Rôle & Features extraites |
| :--- | :--- | :--- | :--- |
| **`contacts`** | `System ID` | 1 ligne / contact | Table pivot : profil, fonctions clés, départements, opt-in communication, statuts CRM. |
| **`adhesion_demission`** | `Organisation - Relation ID` | 1 ligne / organisation | Données historiques d'adhésion : calcul de la cible (`target`), ancienneté et historique de départ. |
| **`meetings`** | `Attendee Relation ID` | 1 ligne / participant / événement | Comportement événementiel historique (`Meeting Status == 'History'`) : assiduité, taux de présence, délais d'inscription, diversité des thématiques (tags). |
| **`click`** | `Relation ID` | 1 ligne / clic | Interactions digitales sur mailings (hors robots) : volume cumulé de clics et récence du dernier clic. |
| **`recipients` + `mailing`** | `Relation ID` / `Mailing ID` | 1 ligne / destinataire / envoi | Engagement emailing (filtré sur les sujets pertinents) : taux d'ouverture, réactivité (< 7 j), clics sur ouverture, récence et taux de rejet/bounce. |


### 2. Définition de la Cible (`target`)

La variable cible est calculée au niveau de l'organisation puis rattachée à chaque contact :
* **`target = True` (Fidèle / Rétention réussie)** : Durée de la dernière adhésion $\ge 3$ ans.
* **`target = False` (Démissionnaire précoce)** : Durée de la dernière adhésion $< 3$ ans **et** année de démission renseignée.
* **`<NA>` (Exclus / Non résolus)** : Membres récents ($< 3$ ans d'ancienneté) toujours actifs.

> **Précaution temporelle** : Pour éviter toute fuite d'information (*data leakage*), les indicateurs de récence (`days_since_*`) des organisations ayant démissionné sont calculés par rapport au **1er janvier de leur année de départ** (et non par rapport à la date du jour).


### 3. Pipeline d'exécution
1. **Chargement & typage** des exports CRM bruts.
2. **Feature Engineering modulaire** via `src.agregation_membres` (calcul vectorisé par individu).
3. **Dédoublonnage & agrégation** à la granularité stricte de 1 ligne par individu (`System ID`).
4. **Jointure finale gauche** sur la population des membres ciblés (`Current Member` et `Former Member`).
5. **Exports & Audit qualité** : génération du fichier Excel et des rapports de distribution `skrub.TableReport`.

## 1. Imports et configuration

In [ ]:
import sys
from pathlib import Path

import pandas as pd
import numpy as np

from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline

import skrub
from skrub import TableVectorizer, tabular_pipeline, TableReport
import skore

# Permet d'importer les modules src/ depuis la racine du projet
sys.path.append(str(Path.cwd().parent))

from src.agregation_membres import (
    add_count_of_a_value,
    days_since_last_event,
    add_unique_meeting_tag_count_by_relation,
    add_delay_within_threshold,
    add_average_delay_by_relation,
    add_rate_condition,
    add_invitation_reactivity_by_relation,
    add_published_registered_flag_by_relation,
    add_number_of_lines,
    add_count_non_empty,
    add_condition_indicator,
    add_rate,
)

## 2. Chargement des données brutes

In [ ]:
# ── Mailing ───────────────────────────────────────────────────────────────
VAR_MAILING = ["Mailing ID", "Mailing tags", "Mailing subject"]
mailing = pd.read_csv(
    "../data/dossier_back_up_mailing/COPIE_mailings_export_2026-01-12.csv",
    sep=";", encoding="latin-1"
)[VAR_MAILING]

mailing_subjects_uniques =pd.read_excel("../data/dossier_back_up_mailing/mailing_subjects_uniques.xlsx")

# ── Clicks ────────────────────────────────────────────────────────────────
VAR_CLICK = ["Mailing ID", "Relation ID", "Bot", "Clicked time"]
click = pd.read_csv(
    "../data/dossier_back_up_mailing/COPIE_clicks_export_2026-01-12.csv",
    sep=";", encoding="latin-1"
)[VAR_CLICK]

# ── Recipients ────────────────────────────────────────────────────────────
VAR_RECIPIENTS = [
    "Mailing ID", "Relation ID", "Datetime Sent",
    "Datetime Viewed (first)", "Datetime Clicked (first)",
    "Datetime Unsubscribed", "Datetime Not received",
]
recipients = pd.read_csv(
    "../data/dossier_back_up_mailing/COPIE_recipients_export_2026-01-12.csv",
    sep=";", encoding="latin-1"
)[VAR_RECIPIENTS]

# ── Contacts ──────────────────────────────────────────────────────────────
VAR_CONTACTS = [
    "Creation date", 
    "System ID", 
    "Company name",
    "Organisation - Relation name",
    "User last online",
    "Communication - Communautés", 
    "Communication - Newsletters",
    "Communication - Partner communications", 
    "Communication - Veille juridique",
    "Company position (description)", "First contact date",
    "Interested in teams", "Meeting&Events speaker",
    "Membership status (description)", "Notification on new comment",
    "Notifications on new post", "Sector of Activities", "Topics.", "VIP",
    "Organisation - Relation ID", "Organisation - Communication - Communautés",
    "Organisation - Communication - Newsletters",
    "Organisation - Communication - Partner communications",
    "Organisation - Communication - Veille juridique",
    "Organisation - Company position", "Organisation - First contact date",
    "Organisation - Membership status", "Organisation - Sector of Activities",
    "Organisation - Topics.", "Function Start date", "Recipient Status",
    "Nr. open invoices", "Amount open invoices",
    "Is CEO / President", "Is deputy managing director", "Is director",
    "Is F LEVEL", "Is M LEVEL", "Is manager", "Is managing director",
    "Is member", "Is responsible for membership (backup)",
    "Is responsible for membership (primary)", "Is subcompany",
    "Is vice president", "Is C LEVEL", "Company subscription",
    "Organisation - Marketing Budget 2019",
    "Department (description)",
    # dummies
    'Direction générale', 
    'Juridique / Fiscal', 
    'RH', 
    'Stratégie / Etudes', 
    'Communication', 
    'Publicité', 
    'RSE', 
    'Affaires Publiques', 
    'Marketing', 
    'Production publicitaire / Création', 
    'Finance', 
    'Marketing Client', 
    'Achats', 
    'Digital', 
    'Commercial', 
    'Marketing Produit',
    'Marketing opérationnel'
]
contacts = pd.read_csv(
    "../data/archive_bdd/COPIE BASE PROCRUSIO CONTACTS DU 09.01.2026.csv",
    sep=";"
)[VAR_CONTACTS]

# ── Meetings ──────────────────────────────────────────────────────────────
VAR_MEETINGS = [
    "Meeting Status", "Meeting Start date", "Meeting Tags", "Meeting Meeting ID",
    "Attendee Is invited?", "Attendee Is registered?", "Attendee Registration date",
    "Attendee Was present?", "Attendee Relation ID",
]
meetings = pd.read_csv(
    "../data/dossier_back_up_procurios_meetings/COPIE_all-attendees-for-meetings-20260112-1725.csv",
    sep=";"
)[VAR_MEETINGS]

# ── Adhesion ──────────────────────────────────────────────────────────────
adhesion_demission_complet = pd.read_excel(
    "../data/adherent/adhesion_demission.xlsx"
).copy()



print("Données chargées :")
print(f"  mailing             : {mailing.shape}")
print(f"  click               : {click.shape}")
print(f"  recipients          : {recipients.shape}")
print(f"  contacts            : {contacts.shape}")
print(f"  meetings            : {meetings.shape}")
print(f"  adhesion_demission  : {adhesion_demission_complet.shape}")

**Enrichissement des données d'adhésion / démission**

On ajoute ici une variable indiquant la durée de la dernière adhésion et un indicateur booléen indiquant si l'adhérent a déjà démissionné avant cette adhésion ou non.

In [ ]:
adhesion_demission_complet_enriched = adhesion_demission_complet.copy()

cols_year = [
    "GROUPE - Année adhésion *",
    "GROUPE - Année démission *",
    "GROUPE - Nombre d'adhésion",
]
for col in cols_year:
    adhesion_demission_complet_enriched[col] = pd.to_numeric(
        adhesion_demission_complet_enriched[col], errors="coerce"
    )

current_year = pd.Timestamp("today").year

adhesion_demission_complet_enriched["duree_derniere_adhesion"] = np.where(
    adhesion_demission_complet_enriched["GROUPE - Année adhésion *"].notna(),
    np.where(
        adhesion_demission_complet_enriched["GROUPE - Année démission *"].notna(),
        adhesion_demission_complet_enriched["GROUPE - Année démission *"]
        - adhesion_demission_complet_enriched["GROUPE - Année adhésion *"],
        current_year - adhesion_demission_complet_enriched["GROUPE - Année adhésion *"],
    ),
    np.nan,
)

adhesion_demission_complet_enriched["a_deja_demissionne"] = (
    adhesion_demission_complet_enriched["GROUPE - Année démission *"].notna()
    | (adhesion_demission_complet_enriched["GROUPE - Nombre d'adhésion"] > 1)
)


**Ajout de la variable cible**

In [ ]:
def create_target_variable(
    df,
    duration_col='duree_derniere_adhesion',
    target_col='target',
    resignation_year_col='GROUPE - Année démission *',
):
    if duration_col not in df.columns:
        raise KeyError(f"Colonne {duration_col!r} introuvable dans le DataFrame")
    if resignation_year_col not in df.columns:
        raise KeyError(f"Colonne {resignation_year_col!r} introuvable dans le DataFrame")

    duration = pd.to_numeric(df[duration_col], errors='coerce')
    resignation_year = df[resignation_year_col]

    target = pd.Series(pd.NA, index=df.index, dtype='boolean')
    mask_long = duration >= 3
    mask_short = duration < 3

    target.loc[mask_long] = True
    target.loc[mask_short & resignation_year.notna()] = False
    # si duration < 3 et Année démission est NaN, target reste NaN

    df[target_col] = target
    return df

adhesion_demission_complet_enriched = create_target_variable(adhesion_demission_complet_enriched, duration_col='duree_derniere_adhesion', target_col='target')
adhesion_demission_complet_enriched[["GROUPE - ID", 'duree_derniere_adhesion', 'target']].sample(20)

In [ ]:
adhesion_demission_complet_enriched = adhesion_demission_complet_enriched[
    [
        "GROUPE - Nom",
        "GROUPE - ID",
        "Organisation - Relation name",
        "Organisation - Relation ID",
        "GROUPE - Année adhésion *",
        "Montant adhésion",
        "GROUPE - Année démission *",
        "Montant démission",
        "GROUPE - Nombre d'adhésion",
        "duree_derniere_adhesion",
        "a_deja_demissionne",
        "target"
    ]
]

**Ajoute à chaque contact l'année de démission de son entreprise**  
Indispensable pour créer les variable days_since...

In [ ]:
# On passe de "adhérent -> démission" à "membre (System ID) -> démission"
contact_to_resignation_year = (
    contacts[["System ID", "Organisation - Relation ID"]]
    .merge(adhesion_demission_complet_enriched, on="Organisation - Relation ID", how="left")
    .drop_duplicates(subset=["System ID"])
    [["System ID", "GROUPE - ID", "GROUPE - Année démission *","target"]]
)

print(f"contact_to_resignation_year : {contact_to_resignation_year.shape}")

## 3. Features Meetings

On travaille uniquement sur `Meeting Status == 'History'` pour capturer le comportement passé. Chaque fonction ajoute une colonne au DataFrame ; on agrège ensuite une seule ligne par `Attendee Relation ID`.

In [ ]:
# Filtre : meetings historiques uniquement
meetings_history = meetings[meetings["Meeting Status"] == "History"].copy()
meetings_history.shape

In [ ]:
meetings_history = meetings_history.merge(
    contact_to_resignation_year.rename(columns={"System ID": "Attendee Relation ID"}),
    on="Attendee Relation ID",
    how="left",
)

# Seulement les vraies inscriptions
meetings_history_registered = meetings_history[
    meetings_history["Attendee Is registered?"].isin(
        ["Registered", "Reserve list", "No reaction"]
    )
].copy()

In [ ]:

# ── Features appliquées séquentiellement ─────────────────────────────────
# Volume & taux de présence
meetings_history = add_count_of_a_value(
    meetings_history,
    id_col="Attendee Relation ID",
    valeur_col="Attendee Is registered?",
    valeur=("Registered", "Reserve list", "No reaction"),
    new_col="registered_count_by_relation"
)
meetings_history = add_count_of_a_value(
    meetings_history,
    id_col="Attendee Relation ID",
    valeur_col="Attendee Was present?",
    valeur=1,
    new_col="present_count_by_relation",
)
meetings_history = add_rate_condition(
    meetings_history,
    id_col="Attendee Relation ID",
    numerateur_mask=meetings_history["Attendee Was present?"] == 1,
    denominateur_mask=~meetings_history["Attendee Is registered?"].isin(
        ["Cancelled", "Opted out"]
    ),
    new_col="presence_rate_by_relation",
)

# Récence
meetings_history_registered = days_since_last_event(
    meetings_history_registered,
    id_col="Attendee Relation ID",
    date_col="Meeting Start date",
    new_col="days_since_last_participation"
)
meetings_history_registered = days_since_last_event(
    meetings_history_registered,
    id_col="Attendee Relation ID",
    date_col="Attendee Registration date",
    new_col="days_since_last_registration",
)

# Comportement qualitatif
meetings_history_registered = add_unique_meeting_tag_count_by_relation(meetings_history_registered)  # diversité thématique
meetings_history_registered = add_average_delay_by_relation(
    meetings_history_registered,
    attendee_relation_id_col="Attendee Relation ID",
    end_date_col="Meeting Start date",
    start_date_col="Attendee Registration date",
    unit="days",
    new_col="average_anticipation_days_by_relation",
)  # délai d'anticipation
meetings_history = add_rate_condition(
    meetings_history,
    id_col="Attendee Relation ID",
    numerateur_mask=(meetings_history["Attendee Is invited?"] == 0) & (meetings_history["Attendee Is registered?"] == "Registered"),
    denominateur_mask=meetings_history["Attendee Is registered?"] == "Registered",
    new_col="spontaneous_participation_rate_by_relation",
)  # participation spontanée
meetings_history = add_invitation_reactivity_by_relation(meetings_history)  # réactivité aux invitations

# Comportements négatifs
meetings_history = add_count_of_a_value(
    meetings_history,
    id_col="Attendee Relation ID",
    valeur_col="Attendee Is registered?",
    valeur="Cancelled",
    new_col="cancelled_count_by_relation",
)  # annulations
meetings_history = add_count_of_a_value(
    meetings_history, 
    id_col="Attendee Relation ID",
    valeur_col="Attendee Is registered?",
    valeur="Opted out", 
    new_col="opted_out_count_by_relation"
)
meetings_history = add_count_of_a_value(
    meetings_history, 
    id_col="Attendee Relation ID",
    valeur_col="Attendee Is registered?",
    valeur="Reserve list", 
    new_col="reserve_list_count_by_relation"
)
meetings_history = add_count_of_a_value(
    meetings_history, 
    id_col="Attendee Relation ID",
    valeur_col="Attendee Is registered?",
    valeur="No reaction", 
    new_col="no_reaction_count_by_relation"
)
meetings_history = add_rate_condition(
    meetings_history,
    id_col="Attendee Relation ID",
    numerateur_mask=(meetings_history["Attendee Is registered?"] == "No reaction") & (meetings_history["Attendee Was present?"] == 1),
    denominateur_mask=meetings_history["Attendee Is registered?"] == "No reaction",
    new_col="no_reaction_attendance_rate_by_relation",
)  # présence malgré No reaction

# Engagement futur (calculé sur tout le dataset, pas seulement History)
published_relation_flags = add_published_registered_flag_by_relation(meetings)

# ── Agrégation : 1 ligne par relation ─────────────────────────────────────
# Agrégation des features "tous statuts"
MEETINGS_FEATURES_ALL = [
    "Attendee Relation ID",
    "registered_count_by_relation",
    "present_count_by_relation",
    "presence_rate_by_relation",
    "spontaneous_participation_rate_by_relation",
    "invitation_reactivity_by_relation",
    "cancelled_count_by_relation",
    "opted_out_count_by_relation",
    "reserve_list_count_by_relation",
    "no_reaction_count_by_relation",
    "no_reaction_attendance_rate_by_relation",
]

meetings_relation_all = (
    meetings_history[MEETINGS_FEATURES_ALL]
    .drop_duplicates(subset=["Attendee Relation ID"])
)

# Agrégation des features "Registered uniquement"
MEETINGS_FEATURES_REGISTERED = [
    "Attendee Relation ID",
    "days_since_last_participation",
    "days_since_last_registration",
    "unique_meeting_tag_count_by_relation",
    "average_anticipation_days_by_relation",
]

meetings_relation_registered = (
    meetings_history_registered[MEETINGS_FEATURES_REGISTERED]
    .drop_duplicates(subset=["Attendee Relation ID"])
)

# fusion
meetings_relation_df = (
    meetings_relation_all
    .merge(
        meetings_relation_registered,
        on="Attendee Relation ID",
        how="left",
    )
    .merge(
        published_relation_flags[
            [
                "Attendee Relation ID",
                "published_and_registered_by_relation"
            ]
        ].drop_duplicates("Attendee Relation ID"),
        on="Attendee Relation ID",
        how="left",
    )
)

print(f"meetings_relation_df : {meetings_relation_df.shape}")
meetings_relation_df.sample(5)

## 4. Features Click

On exclut les clics robots (`Bot == 1`) et les lignes sans `Relation ID`.

In [ ]:
# Filtre : clics humains avec ID connu
click_filtered = click[(click["Bot"] == 0) & (click["Relation ID"].notna())].copy()


In [ ]:
click_filtered = click_filtered.merge(
    contact_to_resignation_year.rename(columns={"System ID": "Relation ID"}),
    on="Relation ID",
    how="left",
)

In [ ]:

# Features
click_filtered = add_number_of_lines(
    click_filtered,
    id_col="Relation ID",
    new_col="clicks_count_by_relation",
)

click_filtered = days_since_last_event(
    click_filtered,
    id_col="Relation ID",
    date_col="Clicked time",
    new_col="days_since_last_click",
)

# Agrégation : 1 ligne par relation
CLICK_FEATURES = ["Relation ID", "clicks_count_by_relation", "days_since_last_click"]
click_relation_df = click_filtered[CLICK_FEATURES].drop_duplicates(subset=["Relation ID"])

print(f"click_relation_df : {click_relation_df.shape}")
click_relation_df.sample(5)

## 5. Features Recipients / Mailing

On joint `recipients` et `mailing` sur `Mailing ID`, puis on calcule les features en deux temps :
- **`non_recus`** : calculé sur *tous* les envois (y compris non-reçus) pour le taux de rejet
- **`recipients_mailing_filtered`** : uniquement les mails bien reçus, pour les features d'engagement

In [ ]:
print(f"mailing avant filtrage : {mailing.shape}")

In [ ]:
# Filtrer la base mailing sur les sujets présents dans mailing_subjects_uniques
mailing = mailing[mailing['Mailing subject'].isin(mailing_subjects_uniques['Mailing subject'])]

print(f"mailing après filtrage : {mailing.shape}")

J'ai donc retiré 8.5 % des mails

In [ ]:
# Jointure recipients × mailing
recipients_mailing = recipients.merge(mailing, on="Mailing ID", how="left")

# Deux périmètres de calcul
has_relation = recipients_mailing["Relation ID"].notna()
recipients_all        = recipients_mailing[has_relation].copy()           # tous envois
recipients_received   = recipients_mailing[
    has_relation & recipients_mailing["Datetime Not received"].isna()
].copy()  # mails reçus uniquement


In [ ]:
recipients_received = recipients_received.merge(
    contact_to_resignation_year.rename(columns={"System ID": "Relation ID"}),
    on="Relation ID",
    how="left",
)

In [ ]:

# ── Features sur les mails reçus (engagement) ────────────────────────────
recipients_received = add_number_of_lines(
    recipients_received,
    id_col="Relation ID",
    new_col="rows_count_by_relation",
)
if "anticipation_less_than_7_days" not in recipients_received.columns:
    recipients_received = add_delay_within_threshold(
        recipients_received,
        end_date_col="Datetime Viewed (first)",
        start_date_col="Datetime Sent",
        threshold_days=7,
        new_col="anticipation_less_than_7_days"
    )
recipients_received = add_count_non_empty(
    recipients_received, 
    id_col='Relation ID',
    colonne="Datetime Viewed (first)", 
    new_col="nb_mails_open_by_relation",
    valid_col="anticipation_less_than_7_days"
)
recipients_received = add_count_non_empty(
    recipients_received, 
    id_col='Relation ID',
    colonne="Datetime Clicked (first)", 
    new_col="nb_mails_clicked_by_relation"
)
recipients_received = add_rate(
    recipients_received,
    id_col='Relation ID',
    numerateur_col="nb_mails_open_by_relation",
    denominateur_col="rows_count_by_relation",
    new_col="taux_ouverture"
)
recipients_received = add_rate(
    recipients_received,
    id_col='Relation ID',
    numerateur_col="nb_mails_clicked_by_relation",
    denominateur_col="rows_count_by_relation",
    new_col="taux_click"
)

tmp = add_count_non_empty(
    recipients_received[recipients_received["Datetime Viewed (first)"].notna() & recipients_received["anticipation_less_than_7_days"]], 
    id_col='Relation ID',
    colonne="Datetime Clicked (first)", 
    new_col="nb_mails_clicked_by_relation_(open)"
)
nb_clic_par_relation = (
    tmp.groupby("Relation ID")["nb_mails_clicked_by_relation_(open)"]
       .first()
)
recipients_received["nb_mails_clicked_by_relation_(open)"] = (
    recipients_received["Relation ID"].map(nb_clic_par_relation)
)
recipients_received = add_rate(
    recipients_received,
    id_col='Relation ID',
    numerateur_col="nb_mails_clicked_by_relation_(open)",
    denominateur_col="nb_mails_open_by_relation",
    new_col="taux_click_sur_ouverture"
)
taux_click_max = recipients_received["taux_click_sur_ouverture"].max()
print(f"taux_click max: {taux_click_max}")

if "anticipation_less_than_7_days" not in recipients_received.columns:
    recipients_received = add_delay_within_threshold(
        recipients_received,
        end_date_col="Datetime Viewed (first)",
        start_date_col="Datetime Sent",
        threshold_days=7,
        new_col="anticipation_less_than_7_days"
    )
recipients_received = days_since_last_event(
    recipients_received,
    id_col="Relation ID",
    date_col="Datetime Viewed (first)",
    new_col="days_since_last_open_by_relation",
    valid_col="anticipation_less_than_7_days"
)
if "anticipation_less_than_7_days" not in recipients_received.columns:
        recipients_received = add_delay_within_threshold(
            recipients_received,
            end_date_col="Datetime Viewed (first)",
            start_date_col="Datetime Sent",
            threshold_days=7,
            new_col="anticipation_less_than_7_days"
        )
recipients_received = add_average_delay_by_relation(
    recipients_received,
    attendee_relation_id_col="Relation ID",
    end_date_col="Datetime Viewed (first)",
    start_date_col="Datetime Sent",
    unit="days",
    new_col="average_days_open_by_relation",
    valid_col="anticipation_less_than_7_days"
)
recipients_received =add_average_delay_by_relation(
    recipients_received,
    attendee_relation_id_col="Relation ID",
    end_date_col="Datetime Viewed (first)",
    start_date_col="Datetime Sent",
    unit="hours",
    valid_col="anticipation_less_than_7_days",
    new_col="average_open_delay_hours",
)
recipients_received = add_condition_indicator(
    recipients_received, 
    id_col="Relation ID",
    condition=recipients_received["Datetime Unsubscribed"].notna() & (recipients_received["Datetime Unsubscribed"] != ''),
    new_col="has_unsubscribed_by_relation",
)
recipients_received = add_unique_meeting_tag_count_by_relation(
    recipients_received,
    attendee_relation_id_col="Relation ID",
    meeting_tags_col="Mailing tags",
    new_col="unique_mailing_tags_count_by_relation",
)

# ── Features sur tous les envois (non-réception) ─────────────────────────
non_recus = add_count_non_empty(
    recipients_all,
    id_col='Relation ID',
    colonne='Datetime Not received',
    new_col='not_received_count_by_relation'
)
# cette variable c'est déjà rows_count_by_relation mais pas dans le bon df?
non_recus = add_number_of_lines(
    recipients_all,
    id_col="Relation ID",
    new_col="total_mail_col",
)

non_recus = add_rate(
    recipients_all,
    id_col='Relation ID',
    numerateur_col="not_received_count_by_relation",
    denominateur_col="total_mail_col",
    new_col="taux_rejet",
)
non_recus_agg = (
    non_recus[["Relation ID", "not_received_count_by_relation", "taux_rejet"]]
    .drop_duplicates(subset=["Relation ID"])
)

# ── Agrégation : 1 ligne par relation ─────────────────────────────────────
RECIPIENTS_FEATURES = [
    "Relation ID",
    "rows_count_by_relation",
    "nb_mails_open_by_relation",
    "taux_ouverture",
    "taux_click",
    "taux_click_sur_ouverture",
    "days_since_last_open_by_relation",
    "average_days_open_by_relation",
    "average_open_delay_hours",
    "has_unsubscribed_by_relation",
    "nb_mails_clicked_by_relation",
    "unique_mailing_tags_count_by_relation",
    # Ajouter ici toute nouvelle feature mailing
]

recipients_mailing_relation_df = (
    recipients_received[RECIPIENTS_FEATURES]
    .drop_duplicates(subset=["Relation ID"])
    .merge(non_recus_agg, on="Relation ID", how="left")
)

print(f"recipients_mailing_relation_df : {recipients_mailing_relation_df.shape}")
recipients_mailing_relation_df.sample(5)

## 6. Jointure finale

On part de `contacts` filtré sur les statuts `Current Member` / `Former Member` (variable cible binaire), puis on y greffe les features des 3 autres sources via `System ID`.

In [ ]:
# Variable cible : membres actuels et anciens uniquement
contacts_filtered = contacts[
    contacts["Organisation - Membership status"].isin(["Current Member", "Former Member"])
].copy()
print(f"contacts filtrés : {contacts_filtered.shape}")

# Jointures successives (left join pour conserver tous les contacts)
df = (
    contacts_filtered
    .merge(click_relation_df, left_on="System ID", right_on="Relation ID", how="left")
    .merge(recipients_mailing_relation_df, left_on="System ID", right_on="Relation ID", how="left", suffixes=("", "_mailing"))
    .merge(meetings_relation_df, left_on="System ID", right_on="Attendee Relation ID", how="left", suffixes=("", "_meetings"))
    .merge(adhesion_demission_complet_enriched,left_on="Organisation - Relation ID",right_on="Organisation - Relation ID",how="left", suffixes=("","_adherent"))
)

print(f"Table finale : {df.shape}")
df.sample(5)

## 7. Export

In [ ]:
OUTPUT_EXCEL = "../data/base_membres.xlsx"
OUTPUT_HTML  = "../outputs/rapports/rapport_base_membres.html"

chunk_size = 30
reports = []

for part, start in enumerate(range(0, len(df.columns), chunk_size), start=1):
    cols = df.columns[start : start + chunk_size]
    chunk_report = TableReport(df[cols])
    output_part_html = OUTPUT_HTML.replace(".html", f"_part{part:02d}.html")
    chunk_report.write_html(output_part_html)
    print(f"Rapport HTML écrit : {output_part_html}")
    reports.append(chunk_report)

report = reports[0]

# Export Excel
df.to_excel(OUTPUT_EXCEL, index=False)
print(f"Export Excel écrit : {OUTPUT_EXCEL}")

report